# Dietary Nitrate/Nitrite Exposure And Oral Microbiome

TRE-side exploratory notebook for testing whether diet-derived nitrate/nitrite/nitroso exposure groups differ in oral microbiome features.

This notebook follows the working `the archived diet/mental-health exploratory workflow` pattern:

- participant data are loaded through `pheno_utils.PhenoLoader` inside TRE;
- diet exposures are built from the **full mega KG** by finding term-matching nodes and traversing back to `hpp_food:*` nodes;
- no 3 GB compound-reference scan is needed;
- intermediate analysis objects are cached as pickle files, not parquet, because some TRE kernels do not have `pyarrow` or `fastparquet`.

## Sections

1. Configuration and imports
2. Diet preprocessing into nitrate/nitrite exposure groups
3. Oral microbiome preprocessing
4. ADA filtering placeholder
5. Confounder loading and configuration
6. Statistical tests
7. Matplotlib plots
8. Summary tables

In [ ]:
# Imports and project setup
from __future__ import annotations

import json
import math
import os
import re
import sys
import time
import warnings
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# If root detection fails inside TRE, set this manually and rerun this cell, e.g.
# MANUAL_PROJECT_ROOT = Path("/home/ec2-user/studies/Diet_Data_Enhancement_Project")
MANUAL_PROJECT_ROOT = None


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = []
    env_root = os.environ.get("DIET_DATA_ENHANCEMENT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        candidates.append(Path(env_root).expanduser())
    if MANUAL_PROJECT_ROOT is not None:
        candidates.append(Path(MANUAL_PROJECT_ROOT).expanduser())
    candidates.extend([start, *start.parents])
    candidates.extend([
        Path.home() / "studies" / "Diet_Data_Enhancement_Project",
        Path.home() / "studies" / "Diet_Data_enhancement",
        Path.home() / "Diet_Data_Enhancement_Project",
        Path.home() / "Diet_Data_enhancement",
    ])
    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (candidate / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js").exists():
            return candidate
        if (candidate / "outputs" / "enhanced_hpp").exists() and (candidate / "downstream_analysis").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT_DIR = PROJECT_ROOT / "downstream_analysis" / "manual" / "nitrate" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)

In [ ]:
# Configuration
ID_COL = "participant_id"
FOOD_COL = "food_id"
REF_FOOD_COL = "hpp_food_id"
GRAMS_COL = "weight_g"

# Full mega KG, matching the archived diet/mental-health exploratory workflow.
KG_MODE = "full_mega"
KG_MAX_HOPS = 4
MEGA_KG_DATA_PATH = PROJECT_ROOT / "outputs" / "visualizations" / "denovo_hpp_kg_mega_flexible_data.js"
FOOD_FEATURE_TABLE = PROJECT_ROOT / "outputs" / "enhanced_hpp" / "1.denovo" / "hpp_feature_matrix_per_100g.csv"

# Persistent caches. Pickle only; no parquet engine required.
FORCE_REBUILD_DIET_EXPOSURES = False
FORCE_REBUILD_ORAL_FEATURES = False
CACHE_DIR = OUTPUT_DIR / "cache_pickle"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATHS = {
    "diet_events": CACHE_DIR / "diet_events.pkl",
    "food_features": CACHE_DIR / "food_features.pkl",
    "kg_nodes": CACHE_DIR / "kg_nodes.pkl",
    "kg_edges": CACHE_DIR / "kg_edges.pkl",
    "kg_reverse_adj": CACHE_DIR / "kg_reverse_adj.pkl",
    "exposure_tables": CACHE_DIR / "exposure_tables.pkl",
    "exposure_summary": CACHE_DIR / "exposure_summary.pkl",
    "oral_features": CACHE_DIR / "oral_features_metaphlan_humann_v2.pkl",
    "confounders": CACHE_DIR / "confounders.pkl",
}

EXPOSURE_SPECS = {
    "vegetable_nitrate": {
        "terms": ["nitrate", "non metal nitrates"],
        "exclude_food_name_tokens": ["smoked", "cured", "sausage", "bacon", "ham", "salami"],
        "source_note": "Full mega KG path-connected nitrate foods, excluding obvious cured/smoked processed terms.",
    },
    "processed_nitrite": {
        "terms": ["nitrite", "sodium nitrite", "sodium nitrate", "non metal nitrites"],
        "include_food_name_tokens": ["smoked", "cured", "sausage", "bacon", "ham", "salami", "hot dog", "processed"],
        "source_note": "Full mega KG path-connected nitrite/preserved food signal.",
    },
    "nitroso_axis": {
        "terms": ["nitroso", "nitrosamine", "nitroso compound"],
        "source_note": "Exploratory full mega KG path-connected nitroso/nitrosamine signal.",
    },
    "arginine_no_axis": {
        "terms": ["arginine", "citrulline", "ornithine", "arginine and proline metabolism"],
        "source_note": "NO-biology support axis from full mega KG/nutrient/pathway terms.",
    },
}

GROUP_METHOD = "tertile"  # tertile, zero_vs_tertile, median
MIN_GROUP_N = 30
FDR_ALPHA = 0.10

GROUP_ORDER = ["low", "mid", "high"]
GROUP_COLORS = {
    "low": "#2166ac",
    "mid": "#f4a582",
    "high": "#b2182b",
    "bottom_10": "#2166ac",
    "top_10": "#b2182b",
}

# One-line confounder control. Add/remove names here, then rerun downward.
ACTIVE_CONFOUNDERS = ["age", "sex", "bmi", "smoking", "alcohol"]
# ACTIVE_CONFOUNDERS = ["age", "sex", "bmi", "smoking", "alcohol", "education", "physical_activity", "sleep_hours"]
# ACTIVE_CONFOUNDERS = []

PRIMARY_ORAL_OUTCOME_PATTERNS = [
    "raw_read_count", "shannon", "simpson", "richness", "alpha",
    "nitrate_reducer", "nitrogen", "nitrate", "nitrite", "nitros",
    "neisseria", "rothia", "veillonella", "actinomyces", "haemophilus", "prevotella", "kingella",
]
MAX_AUTO_ORAL_OUTCOMES = 60


## Helper Functions

In [ ]:
def flatten_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.index, pd.MultiIndex) or out.index.name is not None:
        out = out.reset_index()
    return out.loc[:, ~out.columns.duplicated()].copy()


def normalize_ids(df: pd.DataFrame, id_col: str = ID_COL) -> pd.DataFrame:
    out = flatten_index(df)
    if id_col not in out.columns:
        aliases = ["research_stage_id", "user_id", "RegistrationCode", "participant", "sample_id"]
        for alias in aliases:
            if alias in out.columns:
                out = out.rename(columns={alias: id_col})
                break
    if id_col not in out.columns:
        raise ValueError(f"Could not find participant id column. Columns include: {out.columns.tolist()[:40]}")
    out[id_col] = out[id_col].astype(str)
    return out


def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if path.suffix.lower() in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, low_memory=False)
    if path.suffix.lower() in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="	", low_memory=False)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported table: {path}")


def normalize_token_text(text: object) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def token_contains(text: object, term: object) -> bool:
    clean_text = normalize_token_text(text)
    clean_term = normalize_token_text(term)
    if not clean_text or not clean_term:
        return False
    text_tokens = clean_text.split()
    term_tokens = clean_term.split()
    if len(term_tokens) == 1:
        return term_tokens[0] in text_tokens
    n = len(term_tokens)
    return any(text_tokens[i:i+n] == term_tokens for i in range(len(text_tokens)-n+1))


def contains_any(text: object, terms: list[str]) -> bool:
    return any(token_contains(text, term) for term in terms)


def finite_numeric(s: pd.Series) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna().to_numpy(dtype=float)


def p_adjust_bh(p_values) -> np.ndarray:
    p = np.asarray(pd.to_numeric(pd.Series(p_values), errors="coerce"), dtype=float)
    q = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    out = np.empty(n)
    out[order] = adjusted
    q[valid] = out
    return q


def cliffs_delta(x, y) -> float:
    """Cliff's delta oriented as second group minus first group.

    This matches cohens_d(x, y), which is mean(y) - mean(x).
    Positive values mean the second group tends to have larger values.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    ranks = stats.rankdata(np.concatenate([x, y]))
    rx = ranks[:len(x)].sum()
    u_x = rx - len(x) * (len(x) + 1) / 2
    delta_x_minus_y = (2 * u_x / (len(x) * len(y))) - 1
    return -delta_x_minus_y


def cohens_d(x, y) -> float:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    pooled = math.sqrt(((len(x)-1)*np.var(x, ddof=1) + (len(y)-1)*np.var(y, ddof=1)) / (len(x)+len(y)-2))
    if pooled == 0:
        return np.nan
    return (np.mean(y) - np.mean(x)) / pooled


def cohens_d_ci(d: float, n1: int, n2: int, z: float = 1.96) -> tuple[float, float]:
    if not np.isfinite(d) or n1 < 2 or n2 < 2:
        return np.nan, np.nan
    se = math.sqrt((n1 + n2) / (n1 * n2) + (d**2) / (2 * (n1 + n2 - 2)))
    return d - z * se, d + z * se


# 1. Diet Data Preprocessing Into Exposure Categories

Exposure discovery uses the **full mega KG**, not the compound reference CSVs. We find KG nodes matching nitrate/nitrite/nitroso terms, traverse reverse edges back to `hpp_food:*` nodes, then sum grams of those foods consumed by each participant.

In [ ]:
def load_diet_events_from_phenoloader() -> pd.DataFrame:
    """Load diet events using the same broad strategy as the archived diet/mental-health exploratory workflow."""
    from pheno_utils import PhenoLoader
    from pheno_utils.config import DATASETS_PATH

    pl = PhenoLoader("diet_logging", age_sex_dataset=None, errors="warn")
    dataset_dir = Path(DATASETS_PATH) / pl.dataset
    candidate_paths = [dataset_dir / "diet_logging_events.parquet"]

    if "diet_logging_events" in getattr(pl, "dfs", {}):
        return normalize_ids(flatten_index(pl.dfs["diet_logging_events"]))

    if "diet_logging" in getattr(pl, "dfs", {}):
        summary_df = flatten_index(pl.dfs["diet_logging"])
        if "diet_logging_events" in summary_df.columns:
            for rel_path in summary_df["diet_logging_events"].dropna().astype(str).unique()[:20]:
                candidate_paths.append(dataset_dir / rel_path)

    events_path = next((path for path in candidate_paths if path.exists()), None)
    if events_path is None:
        PLACEHOLDER
    print("Reading diet events:", events_path, flush=True)
    return normalize_ids(pd.read_parquet(events_path))


def js_value_after_key(text: str, key: str):
    marker = f"{key}:"
    start = text.index(marker) + len(marker)
    decoder = json.JSONDecoder()
    value, _end = decoder.raw_decode(text[start:])
    return value


def load_mega_kg_data(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    if not path.exists():
        raise FileNotFoundError(f"Mega KG data file not found: {path}")
    print("Loading full mega KG:", path, flush=True)
    text = path.read_text(encoding="utf-8")
    kinds = js_value_after_key(text, "kinds")
    relations = js_value_after_key(text, "relations")
    compact_nodes = js_value_after_key(text, "nodes")
    compact_edges = js_value_after_key(text, "edges")

    nodes = pd.DataFrame(compact_nodes, columns=["key", "label", "kind_idx", "x", "y"])
    nodes["kind"] = nodes["kind_idx"].map(lambda i: kinds[int(i)])
    nodes = nodes[["key", "kind", "label"]]

    key_by_idx = nodes["key"].astype(str).to_numpy()
    edges = pd.DataFrame(compact_edges, columns=["source_idx", "target_idx", "relation_idx"])
    edges["source"] = key_by_idx[edges["source_idx"].astype(int).to_numpy()]
    edges["target"] = key_by_idx[edges["target_idx"].astype(int).to_numpy()]
    edges["relation"] = edges["relation_idx"].map(lambda i: relations[int(i)])
    edges = edges[["source", "target", "relation"]]
    return nodes, edges


def build_reverse_adjacency(kg_edges: pd.DataFrame) -> dict[str, set[str]]:
    reverse_adj: dict[str, set[str]] = {}
    for source, target in kg_edges[["source", "target"]].dropna().astype(str).itertuples(index=False):
        reverse_adj.setdefault(target, set()).add(source)
    return reverse_adj


def kg_target_nodes_for_terms(kg_nodes: pd.DataFrame, terms: list[str]) -> set[str]:
    target_nodes = set()
    search_cols = [c for c in ["key", "label", "kind"] if c in kg_nodes.columns]
    for row in kg_nodes[search_cols].fillna("").astype(str).itertuples(index=False, name=None):
        row_text = " | ".join(row)
        if any(token_contains(row_text, term) for term in terms):
            target_nodes.add(str(row[0]))
    return target_nodes


def hpp_food_id_from_node(node: str) -> str | None:
    node = str(node)
    if node.startswith("hpp_food:"):
        return node.split(":", 1)[1]
    return None


def kg_path_connected_food_ids(kg_nodes, kg_edges, reverse_adj, terms, max_hops=KG_MAX_HOPS) -> tuple[set[str], dict]:
    target_nodes = kg_target_nodes_for_terms(kg_nodes, terms)
    frontier = set(target_nodes)
    visited = set(target_nodes)
    food_ids = set()
    for depth in range(max_hops + 1):
        for node in frontier:
            food_id = hpp_food_id_from_node(node)
            if food_id is not None:
                food_ids.add(food_id)
        if depth == max_hops:
            break
        next_frontier = set()
        for node in frontier:
            next_frontier.update(reverse_adj.get(node, set()))
        next_frontier -= visited
        visited.update(next_frontier)
        frontier = next_frontier
        if not frontier:
            break
    return food_ids, {
        "kg_mode": KG_MODE,
        "target_node_count": len(target_nodes),
        "visited_node_count": len(visited),
        "connected_food_count": len(food_ids),
        "max_hops": max_hops,
        "example_target_nodes": " | ".join(sorted(target_nodes)[:15]),
    }


def load_food_features() -> pd.DataFrame:
    food = read_table(FOOD_FEATURE_TABLE)
    food[REF_FOOD_COL] = food[REF_FOOD_COL].astype(str)
    return food


def filter_connected_foods(food_ids: set[str], food_features: pd.DataFrame, spec: dict) -> set[str]:
    if not food_ids:
        return set()
    ref = food_features[food_features[REF_FOOD_COL].astype(str).isin({str(x) for x in food_ids})].copy()
    text_cols = [c for c in ["hpp_food_name", "hpp_product_name", "hpp_short_description", "hpp_category", "canonical_name", "canonical_category"] if c in ref.columns]
    if not text_cols:
        return {str(x) for x in food_ids}
    name_text = ref[text_cols].fillna("").astype(str).agg(" | ".join, axis=1)
    include = spec.get("include_food_name_tokens") or []
    exclude = spec.get("exclude_food_name_tokens") or []
    keep = pd.Series(True, index=ref.index)
    if include:
        keep &= name_text.map(lambda v: contains_any(v, include))
    if exclude:
        keep &= ~name_text.map(lambda v: contains_any(v, exclude))
    return set(ref.loc[keep, REF_FOOD_COL].astype(str))


def participant_exposure_from_kg_connected_foods(diet_events: pd.DataFrame, connected_food_ids: set[str], exposure_name: str) -> pd.DataFrame:
    food_col = FOOD_COL if FOOD_COL in diet_events.columns else "hpp_food_id"
    grams_col = GRAMS_COL if GRAMS_COL in diet_events.columns else "grams_consumed"
    d = diet_events[[ID_COL, food_col, grams_col]].copy()
    d[food_col] = d[food_col].astype(str)
    d[grams_col] = pd.to_numeric(d[grams_col], errors="coerce").fillna(0)
    connected = {str(x) for x in connected_food_ids}
    d["is_connected_food"] = d[food_col].isin(connected)
    d["exposure_value"] = np.where(d["is_connected_food"], d[grams_col], 0.0)
    out = d.groupby(ID_COL, as_index=False).agg(
        exposure_value=("exposure_value", "sum"),
        connected_food_events=("is_connected_food", "sum"),
        food_events=(food_col, "count"),
        unique_foods=(food_col, "nunique"),
        total_logged_g=(grams_col, "sum"),
    )
    out["exposure_per_1000g_logged"] = np.where(out["total_logged_g"] > 0, out["exposure_value"] / out["total_logged_g"] * 1000.0, np.nan)
    out["log1p_exposure_value"] = np.log1p(out["exposure_value"])
    out["exposure"] = exposure_name
    return out


def assign_exposure_groups(values: pd.Series, method: str = GROUP_METHOD) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").fillna(0)
    if method == "zero_vs_tertile":
        out = pd.Series("zero", index=v.index, dtype="object")
        positive = v[v > 0]
        if positive.empty:
            return out
        q1, q2 = positive.quantile([1/3, 2/3]).to_numpy()
        out[(v > 0) & (v <= q1)] = "low"
        out[(v > q1) & (v <= q2)] = "mid"
        out[v > q2] = "high"
        return out
    if method == "median":
        med = v.median()
        return pd.Series(np.where(v <= med, "low", "high"), index=v.index)
    positive = v[v > 0]
    out = pd.Series("low", index=v.index, dtype="object")
    if positive.nunique() < 3:
        out[v > 0] = "high"
        return out
    q1, q2 = positive.quantile([1/3, 2/3]).to_numpy()
    out[(v > q1) & (v <= q2)] = "mid"
    out[v > q2] = "high"
    return out


def assign_decile_groups(values: pd.Series) -> pd.Series:
    v = pd.to_numeric(values, errors="coerce").fillna(0)
    lo, hi = v.quantile([0.10, 0.90]).to_numpy()
    out = pd.Series(np.nan, index=v.index, dtype="object")
    out[v <= lo] = "bottom_10"
    out[v >= hi] = "top_10"
    return out

In [ ]:
# Build or load diet-derived exposure tables.
start = time.time()
if (not FORCE_REBUILD_DIET_EXPOSURES) and CACHE_PATHS["exposure_tables"].exists() and CACHE_PATHS["exposure_summary"].exists():
    print("Loading cached diet exposures:", CACHE_PATHS["exposure_tables"], flush=True)
    exposure_tables = pd.read_pickle(CACHE_PATHS["exposure_tables"])
    exposure_summary = pd.read_pickle(CACHE_PATHS["exposure_summary"])
    diet_events = pd.read_pickle(CACHE_PATHS["diet_events"]) if CACHE_PATHS["diet_events"].exists() else None
    food_features = pd.read_pickle(CACHE_PATHS["food_features"]) if CACHE_PATHS["food_features"].exists() else None
else:
    print("[1/5] Loading diet events via PhenoLoader...", flush=True)
    diet_events = load_diet_events_from_phenoloader()
    diet_events.to_pickle(CACHE_PATHS["diet_events"])
    print("diet_events", diet_events.shape, f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[2/5] Loading food features...", flush=True)
    food_features = load_food_features()
    food_features.to_pickle(CACHE_PATHS["food_features"])
    print("food_features", food_features.shape, f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[3/5] Loading full mega KG...", flush=True)
    if CACHE_PATHS["kg_nodes"].exists() and CACHE_PATHS["kg_edges"].exists():
        kg_nodes = pd.read_pickle(CACHE_PATHS["kg_nodes"])
        kg_edges = pd.read_pickle(CACHE_PATHS["kg_edges"])
    else:
        kg_nodes, kg_edges = load_mega_kg_data(MEGA_KG_DATA_PATH)
        kg_nodes.to_pickle(CACHE_PATHS["kg_nodes"])
        kg_edges.to_pickle(CACHE_PATHS["kg_edges"])
    print("kg_nodes", kg_nodes.shape, "kg_edges", kg_edges.shape, f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[4/5] Building reverse adjacency...", flush=True)
    if CACHE_PATHS["kg_reverse_adj"].exists():
        kg_reverse_adj = pd.read_pickle(CACHE_PATHS["kg_reverse_adj"])
    else:
        kg_reverse_adj = build_reverse_adjacency(kg_edges)
        pd.to_pickle(kg_reverse_adj, CACHE_PATHS["kg_reverse_adj"])
    print("reverse adjacency targets", len(kg_reverse_adj), f"elapsed={time.time()-start:.1f}s", flush=True)

    print("[5/5] Building exposure tables from full KG paths...", flush=True)
    exposure_tables = {}
    exposure_summaries = []
    for i, (exposure_name, spec) in enumerate(EXPOSURE_SPECS.items(), start=1):
        step = time.time()
        print(f"  exposure {i}/{len(EXPOSURE_SPECS)}: {exposure_name}", flush=True)
        raw_connected, kg_detail = kg_path_connected_food_ids(
            kg_nodes, kg_edges, kg_reverse_adj, spec.get("terms", [exposure_name]), max_hops=KG_MAX_HOPS
        )
        connected = filter_connected_foods(raw_connected, food_features, spec)
        exposure = participant_exposure_from_kg_connected_foods(diet_events, connected, exposure_name)
        exposure["exposure_group"] = assign_exposure_groups(exposure["exposure_value"], method=GROUP_METHOD)
        exposure["decile_group"] = assign_decile_groups(exposure["exposure_value"])
        exposure_tables[exposure_name] = exposure
        detail = {
            "exposure": exposure_name,
            "source": "full_mega_kg_path_connected_food_grams",
            "terms": ", ".join(spec.get("terms", [])),
            "raw_connected_food_count": len(raw_connected),
            "connected_food_count_after_name_filters": len(connected),
            "participants": int(exposure[ID_COL].nunique()),
            "nonzero_participants": int((exposure["exposure_value"] > 0).sum()),
            "group_counts": exposure["exposure_group"].value_counts(dropna=False).to_dict(),
            "decile_counts": exposure["decile_group"].value_counts(dropna=False).to_dict(),
            **kg_detail,
        }
        exposure_summaries.append(detail)
        print(f"    connected={len(connected):,}, participants={detail['participants']:,}, nonzero={detail['nonzero_participants']:,}, elapsed={time.time()-step:.1f}s", flush=True)

    exposure_summary = pd.DataFrame(exposure_summaries)
    exposure_summary.to_pickle(CACHE_PATHS["exposure_summary"])
    pd.to_pickle(exposure_tables, CACHE_PATHS["exposure_tables"])
    print(f"Done exposure build. elapsed={time.time()-start:.1f}s", flush=True)

exposure_summary

In [ ]:
# Inspect connected foods for a selected exposure.
SELECTED_EXPOSURE_TO_INSPECT = "vegetable_nitrate"
if food_features is None and CACHE_PATHS["food_features"].exists():
    food_features = pd.read_pickle(CACHE_PATHS["food_features"])
exp = exposure_tables[SELECTED_EXPOSURE_TO_INSPECT]
connected_ids = set(exp.loc[exp["exposure_value"] > 0, ID_COL].astype(str))
# This table shows KG-connected food definitions, not participant rows.
# Recompute connected foods for inspection if KG objects are in memory/cache.
kg_nodes = pd.read_pickle(CACHE_PATHS["kg_nodes"])
kg_edges = pd.read_pickle(CACHE_PATHS["kg_edges"])
kg_reverse_adj = pd.read_pickle(CACHE_PATHS["kg_reverse_adj"])
spec = EXPOSURE_SPECS[SELECTED_EXPOSURE_TO_INSPECT]
raw_connected, _ = kg_path_connected_food_ids(kg_nodes, kg_edges, kg_reverse_adj, spec.get("terms", []), max_hops=KG_MAX_HOPS)
connected = filter_connected_foods(raw_connected, food_features, spec)
cols = [c for c in [REF_FOOD_COL, "hpp_food_name", "hpp_product_name", "hpp_category", "canonical_name", "canonical_category"] if c in food_features.columns]
food_features[food_features[REF_FOOD_COL].astype(str).isin(connected)][cols].head(100)

# 2. Oral Microbiome Preprocessing

This loads oral microbiome through `PhenoLoader('oral_microbiome')`. It first uses the main table (`pl['oral_microbiome']`) for directly available numeric outcomes and then tries to load linked MetaPhlAn/HUMAnN bulk files if paths are exposed in that table. If bulk pathway/taxon files are not available in your TRE mount, the notebook still proceeds with main-table oral metrics such as read count and any numeric columns present.

In [ ]:
NITRATE_REDUCER_PATTERNS = {
    "actinomyces": r"actinomyces",
    "haemophilus": r"haemophilus",
    "kingella": r"kingella",
    "neisseria": r"neisseria",
    "prevotella": r"prevotella",
    "rothia": r"rothia",
    "veillonella": r"veillonella",
}
NITROGEN_PATHWAY_RE = re.compile(r"nitrate|nitrite|nitric|nitros|denitrification|nitrogen|\bnar[A-Z]?\b|\bnir[A-Z]?\b|\bnor[A-Z]?\b|\bnos[Z]?\b", re.I)

# Confirmed in TRE from PhenoLoader('oral_microbiome'):
# - MetaPhlAn aggregated genus/species/family parquet files load locally.
# - HUMAnN aggregated pathway abundance/coverage pathway-level arrow files load locally.
# - HUMAnN microbe-level path columns point to s3:// per-sample files and are skipped by default.
ORAL_BULK_SPECS = [
    ("oral_metaphlan_genus", r"^metaphlan_abundance_genus_parquet$", "taxa"),
    ("oral_metaphlan_species", r"^metaphlan_abundance_species_parquet$", "taxa"),
    ("oral_metaphlan_family", r"^metaphlan_abundance_family_parquet$", "taxa"),
    ("oral_humann_pathway_abundance_pathway_level", r"^humann_aggregated_pathway_abundance_pathway_level_arrow$|^humann_pathway_abundance_pathway_level_parquet$", "pathway"),
    ("oral_humann_pathway_coverage_pathway_level", r"^humann_aggregated_pathway_coverage_pathway_level_arrow$|^humann_pathway_coverage_pathway_level_parquet$", "pathway"),
]

ORAL_METADATA_COLS = {ID_COL, "cohort", "research_stage", "array_index", "collection_date"}

OPTIONAL_ORAL_BULK_SPECS_NOT_DEFAULT = [
    ("oral_humann_pathway_abundance_microbe_level", r"humann.*pathway.*abundance.*microbe.*level.*(parquet|arrow)", "pathway_microbe_level"),
    ("oral_humann_pathway_coverage_microbe_level", r"humann.*pathway.*coverage.*microbe.*level.*(parquet|arrow)", "pathway_microbe_level"),
    ("oral_humann_gene_families_gene_level", r"humann.*gene.*families.*gene.*level.*(parquet|arrow)", "gene"),
    ("oral_humann_gene_families_microbe_level", r"humann.*gene.*families.*microbe.*level.*(parquet|arrow)", "gene_microbe_level"),
]


def make_phenoloader(dataset: str):
    from pheno_utils import PhenoLoader
    return PhenoLoader(dataset, errors="warn")


def phenoloader_dataset_dir(pl) -> Path | None:
    try:
        from pheno_utils.config import DATASETS_PATH
        return Path(DATASETS_PATH) / pl.dataset
    except Exception as exc:
        print(f"Could not infer DATASETS_PATH for {getattr(pl, 'dataset', 'unknown')}: {exc}", flush=True)
        return None


def flatten_oral_table(df: pd.DataFrame) -> pd.DataFrame:
    out = flatten_index(df)
    out.columns = make_unique_columns(out.columns)
    return out


def make_unique_columns(columns) -> list[str]:
    seen = {}
    unique = []
    for col in columns:
        base = str(col)
        count = seen.get(base, 0)
        unique.append(base if count == 0 else f"{base}__dup{count}")
        seen[base] = count + 1
    return unique


def numeric_series_from_column(df: pd.DataFrame, col: str) -> pd.Series:
    values = df.loc[:, col]
    if isinstance(values, pd.DataFrame):
        values = values.iloc[:, 0]
    return pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)


def pheno_main_table(dataset: str, table: str | None = None, return_loader: bool = False):
    pl = make_phenoloader(dataset)
    table = table or dataset
    if table in getattr(pl, "dfs", {}):
        df = normalize_ids(flatten_oral_table(pl.dfs[table]))
    else:
        try:
            df = normalize_ids(flatten_oral_table(pl[table]))
        except Exception as exc:
            available = list(getattr(pl, "dfs", {}).keys())
            raise KeyError(f"Could not load {dataset}/{table}. Available pl.dfs: {available}") from exc
    return (df, pl) if return_loader else df


def choose_participant_id_col(df: pd.DataFrame) -> str | None:
    for col in [ID_COL, "participant_id", "sample_id", "sample_name", "wgs_dna_code", "SampleID", "sample"]:
        if col in df.columns:
            return col
    return None


def sanitize_feature_name(value: object) -> str:
    text = str(value).replace("|", "_")
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_").lower()
    return text[:180] if text else "feature"


def read_bulk_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False)
    if suffix in {".tsv", ".txt"}:
        return pd.read_csv(path, sep="\t", low_memory=False)
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix in {".arrow", ".feather"}:
        try:
            return pd.read_feather(path)
        except Exception:
            import pyarrow.feather as feather
            return feather.read_feather(path).to_pandas()
    raise ValueError(f"Unsupported bulk file extension: {path}")


def candidate_bulk_paths(raw: str, dataset_dir: Path | None) -> list[Path]:
    raw = str(raw).strip()
    if not raw or raw.lower() in {"nan", "none"} or raw.startswith("s3://"):
        return []
    raw_path = Path(raw)
    cleaned = raw.lstrip("./")
    candidates = [raw_path]
    if dataset_dir is not None:
        candidates.extend([
            dataset_dir / raw,
            dataset_dir / cleaned,
            dataset_dir / "oral_microbiome" / raw,
            dataset_dir / "oral_microbiome" / cleaned,
        ])
    return list(dict.fromkeys(candidates))


def load_optional_bulk_table_from_paths(primary: pd.DataFrame, path_regex: str, dataset_dir: Path | None, max_unique_paths: int = 20) -> tuple[pd.DataFrame | None, Path | None]:
    rx = re.compile(path_regex, re.I)
    path_cols = [c for c in primary.columns if rx.search(str(c))]
    if not path_cols:
        print(f"No oral bulk path column matched: {path_regex}", flush=True)
        return None, None

    attempted = []
    for col in path_cols:
        for raw in primary[col].dropna().astype(str).unique():
            attempted.append(raw)
            for candidate in candidate_bulk_paths(raw, dataset_dir):
                if not candidate.exists():
                    continue
                try:
                    table = flatten_oral_table(read_bulk_file(candidate))
                    print(f"Loaded oral bulk {col}: {candidate} shape={table.shape}", flush=True)
                    return table, candidate
                except Exception as exc:
                    print(f"Could not read oral bulk {candidate}: {exc}", flush=True)
            if len(attempted) >= max_unique_paths:
                break
        if len(attempted) >= max_unique_paths:
            break
    print(f"Bulk column(s) existed but no readable local file was found for {path_regex}. Example values: {attempted[:3]}", flush=True)
    return None, None


def normalize_long_microbiome_table(df: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    df = flatten_oral_table(df)
    id_col = choose_participant_id_col(df)
    feature_candidates = [c for c in df.columns if re.search(r"feature|taxon|clade|pathway|gene|name|species|genus", str(c), re.I)]
    value_candidates = []
    for c in df.columns:
        if c == id_col or c in ORAL_METADATA_COLS:
            continue
        vals = pd.to_numeric(df.loc[:, c], errors="coerce")
        if vals.notna().sum() > 0:
            value_candidates.append(c)
    if id_col is None or not feature_candidates or not value_candidates:
        return None
    feature_col = feature_candidates[0]
    value_col = next((c for c in value_candidates if re.search(r"abundance|coverage|relative|value", str(c), re.I)), value_candidates[0])
    long_df = df[[id_col, feature_col]].copy()
    long_df[value_col] = pd.to_numeric(df.loc[:, value_col], errors="coerce")
    long_df = long_df.dropna(subset=[id_col, feature_col, value_col])
    if long_df.empty:
        return None
    wide = long_df.pivot_table(index=id_col, columns=feature_col, values=value_col, aggfunc="mean").reset_index()
    wide = wide.rename(columns={id_col: ID_COL})
    return normalize_wide_microbiome_table(wide, prefix)


def normalize_transposed_microbiome_table(original: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    candidate = original.copy()
    if isinstance(candidate.index, pd.MultiIndex):
        return None
    participant_like_cols = pd.Series(candidate.columns).astype(str).str.fullmatch(r"\d{5,}").mean()
    if participant_like_cols < 0.25:
        return None
    transposed = candidate.T.reset_index().rename(columns={"index": ID_COL})
    return normalize_wide_microbiome_table(transposed, prefix)


def normalize_wide_microbiome_table(df: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    df = flatten_oral_table(df)
    id_col = choose_participant_id_col(df)
    if id_col is None:
        return None
    df = df.rename(columns={id_col: ID_COL})
    df = flatten_oral_table(df)
    feature_cols = [c for c in df.columns if c not in ORAL_METADATA_COLS]

    numeric_parts = {}
    for c in feature_cols:
        vals = pd.to_numeric(df.loc[:, c], errors="coerce")
        if vals.notna().sum() > 0:
            numeric_parts[f"{prefix}_{sanitize_feature_name(c)}"] = vals.to_numpy()
    if not numeric_parts:
        return None

    out = pd.concat(
        [pd.DataFrame({ID_COL: df[ID_COL].astype(str).to_numpy()}), pd.DataFrame(numeric_parts, index=df.index)],
        axis=1,
    )
    out.columns = make_unique_columns(out.columns)
    out = out.copy()
    return out.groupby(ID_COL, as_index=False).mean(numeric_only=True)


def normalize_microbiome_bulk_table(df: pd.DataFrame, prefix: str) -> tuple[pd.DataFrame | None, str | None]:
    attempts = [
        ("wide", normalize_wide_microbiome_table),
        ("long", normalize_long_microbiome_table),
        ("transposed", normalize_transposed_microbiome_table),
    ]
    for mode, func in attempts:
        try:
            wide = func(df, prefix)
            if wide is not None and wide.shape[1] > 1:
                print(f"Normalized {prefix} as {mode}: {wide.shape[0]} participants, {wide.shape[1] - 1} numeric features", flush=True)
                return wide, mode
        except Exception as exc:
            print(f"Normalize attempt failed for {prefix} ({mode}): {exc}", flush=True)
    print(f"Could not normalize {prefix}; columns include: {flatten_oral_table(df).columns.tolist()[:20]}", flush=True)
    return None, None


def alpha_diversity_from_matrix(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    matrix = flatten_oral_table(matrix)
    features = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL and not c.endswith("matched_feature_count")]
    if not features:
        return matrix[[ID_COL]].copy()
    x = matrix[features].clip(lower=0).fillna(0).to_numpy(dtype=float)
    row_sums = x.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.divide(x, row_sums[:, None], out=np.zeros_like(x), where=row_sums[:, None] > 0)
        logp = np.where(p > 0, np.log(p), 0)
    return pd.DataFrame({
        ID_COL: matrix[ID_COL].astype(str).to_numpy(),
        f"{prefix}_shannon": -(p * logp).sum(axis=1),
        f"{prefix}_simpson": 1 - (p ** 2).sum(axis=1),
        f"{prefix}_richness": (x > 0).sum(axis=1),
        f"{prefix}_total_abundance": row_sums,
    })


def nitrate_reducer_from_matrix(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame:
    matrix = flatten_oral_table(matrix)
    numeric_cols = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    out = pd.DataFrame({ID_COL: matrix[ID_COL].astype(str)})
    matched_all = []
    for name, pattern in NITRATE_REDUCER_PATTERNS.items():
        matches = [c for c in numeric_cols if re.search(pattern, c, re.I)]
        summed = matrix[matches].sum(axis=1) if matches else pd.Series(0.0, index=matrix.index)
        out[f"{prefix}_nitrate_reducer_{name}"] = summed.to_numpy()
        out[f"{prefix}_nitrate_reducer_{name}_present"] = (summed > 0).astype(float).to_numpy()
        matched_all.extend(matches)
        print(f"{prefix} {name}: matched {len(matches)} features", flush=True)
    matched_all = sorted(set(matched_all))
    total = matrix[matched_all].sum(axis=1) if matched_all else pd.Series(0.0, index=matrix.index)
    out[f"{prefix}_nitrate_reducer_total"] = total.to_numpy()
    out[f"{prefix}_nitrate_reducer_present_any"] = (total > 0).astype(float).to_numpy()
    out[f"{prefix}_nitrate_reducer_matched_feature_count"] = len(matched_all)
    return out


def nitrogen_pathway_features(matrix: pd.DataFrame, prefix: str) -> pd.DataFrame | None:
    matrix = flatten_oral_table(matrix)
    numeric_cols = [c for c in matrix.select_dtypes(include="number").columns if c != ID_COL]
    nitrogen_cols = [c for c in numeric_cols if NITROGEN_PATHWAY_RE.search(c)]
    print(f"{prefix}: matched {len(nitrogen_cols)} nitrogen pathway/gene columns", flush=True)
    if not nitrogen_cols:
        return None
    out = matrix[[ID_COL] + nitrogen_cols].copy()
    total = out[nitrogen_cols].sum(axis=1)
    out[f"{prefix}_nitrogen_pathway_total"] = total.to_numpy()
    out[f"{prefix}_nitrogen_pathway_present_any"] = (total > 0).astype(float).to_numpy()
    out[f"{prefix}_nitrogen_pathway_matched_feature_count"] = len(nitrogen_cols)
    out.columns = make_unique_columns(out.columns)
    return out.copy()


def top_microbiome_features(wide: pd.DataFrame, prefix: str, n: int = 20) -> pd.DataFrame:
    wide = flatten_oral_table(wide)
    numeric_cols = [c for c in wide.select_dtypes(include="number").columns if c != ID_COL]
    scores = []
    for col in numeric_cols:
        x = numeric_series_from_column(wide, col)
        if x.notna().sum() < MIN_GROUP_N:
            continue
        prevalence = float((x.fillna(0) > 0).mean())
        variance = float(np.nanvar(np.log1p(x.clip(lower=0))))
        mean_abundance = float(np.nanmean(x))
        reducer_bonus = 10.0 if any(re.search(pat, col, re.I) for pat in NITRATE_REDUCER_PATTERNS.values()) else 0.0
        scores.append((reducer_bonus + variance, prevalence, mean_abundance, col))
    chosen = [col for *_rest, col in sorted(scores, reverse=True)[:n]]
    print(f"{prefix}: selected {len(chosen)} top variable features", flush=True)
    return wide[[ID_COL] + chosen].copy() if chosen else wide[[ID_COL]].copy()


def build_oral_features() -> pd.DataFrame:
    primary, pl = pheno_main_table("oral_microbiome", "oral_microbiome", return_loader=True)
    dataset_dir = phenoloader_dataset_dir(pl)
    print("oral primary", primary.shape, flush=True)
    print("oral dataset_dir", dataset_dir, flush=True)

    numeric_cols = [c for c in primary.select_dtypes(include="number").columns if c != ID_COL]
    keep_numeric = [c for c in numeric_cols if any(token_contains(c, pat) for pat in PRIMARY_ORAL_OUTCOME_PATTERNS)]
    if not keep_numeric and "raw_read_count" in primary.columns:
        keep_numeric = ["raw_read_count"]
    oral_parts = [primary[[ID_COL] + keep_numeric].copy()] if keep_numeric else [primary[[ID_COL]].copy()]
    loaded_oral_tables = []

    for prefix, path_regex, feature_kind in ORAL_BULK_SPECS:
        bulk, bulk_path = load_optional_bulk_table_from_paths(primary, path_regex, dataset_dir)
        if bulk is None:
            print(f"optional bulk not found/readable for {prefix}; continuing", flush=True)
            continue
        wide, mode = normalize_microbiome_bulk_table(bulk, prefix)
        if wide is None:
            print(f"optional bulk found but could not normalize for {prefix}; continuing", flush=True)
            continue
        loaded_oral_tables.append({"prefix": prefix, "kind": feature_kind, "mode": mode, "path": str(bulk_path), "participants": wide.shape[0], "features": wide.shape[1] - 1})

        if feature_kind == "taxa":
            oral_parts.append(alpha_diversity_from_matrix(wide, prefix))
            oral_parts.append(nitrate_reducer_from_matrix(wide, prefix))
            oral_parts.append(top_microbiome_features(wide, prefix, n=20))
        elif feature_kind == "pathway":
            nitrogen = nitrogen_pathway_features(wide, prefix)
            if nitrogen is not None:
                oral_parts.append(nitrogen)
            oral_parts.append(top_microbiome_features(wide, prefix, n=20))

    oral = oral_parts[0]
    for part in oral_parts[1:]:
        oral = oral.merge(flatten_oral_table(part), on=ID_COL, how="outer")
    oral = flatten_oral_table(oral)
    if loaded_oral_tables:
        print("Loaded oral bulk summary:", flush=True)
        display(pd.DataFrame(loaded_oral_tables))
    return normalize_ids(oral)


if CACHE_PATHS["oral_features"].exists() and not FORCE_REBUILD_ORAL_FEATURES:
    oral_features = pd.read_pickle(CACHE_PATHS["oral_features"])
    print("Loaded cached oral_features", oral_features.shape)
else:
    oral_features = build_oral_features()
    oral_features.to_pickle(CACHE_PATHS["oral_features"])
    print("Built oral_features", oral_features.shape)

print("Oral feature columns selected from preprocessing:")
print(pd.Series(oral_features.columns).head(180).to_string(index=False))
oral_features.head()


In [ ]:

def select_oral_outcomes(oral: pd.DataFrame) -> list[str]:
    numeric_cols = [c for c in oral.select_dtypes(include="number").columns if c != ID_COL]
    numeric_cols = [c for c in numeric_cols if not c.endswith("matched_feature_count")]
    if not numeric_cols:
        return []

    priority_patterns = [
        "shannon", "simpson", "richness",
        "nitrate_reducer_total", "nitrate_reducer_present_any",
        "nitrate_reducer_neisseria", "nitrate_reducer_rothia", "nitrate_reducer_veillonella",
        "nitrate_reducer_actinomyces", "nitrate_reducer_haemophilus", "nitrate_reducer_prevotella", "nitrate_reducer_kingella",
        "neisseria", "rothia",
        "nitrogen_pathway_total", "nitrate", "nitrite", "nitros",
    ]
    selected = []
    for pat in priority_patterns:
        for col in numeric_cols:
            if col not in selected and re.search(pat, col, re.I):
                selected.append(col)

    # Add the most variable individual genus/species/pathway abundances so this is not limited to QC read count.
    remaining = [c for c in numeric_cols if c not in selected and c != "raw_read_count"]
    scored = []
    for col in remaining:
        x = pd.to_numeric(oral[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if x.notna().sum() < MIN_GROUP_N:
            continue
        prevalence = float((x.fillna(0) > 0).mean())
        variance = float(np.nanvar(np.log1p(x.clip(lower=0))))
        scored.append((variance, prevalence, col))
    selected.extend([col for *_score, col in sorted(scored, reverse=True)[: max(0, MAX_AUTO_ORAL_OUTCOMES - len(selected) - 1)]])

    if "raw_read_count" in numeric_cols:
        selected.append("raw_read_count")  # Keep QC last, not as the primary biological endpoint.
    return selected[:MAX_AUTO_ORAL_OUTCOMES]

ORAL_OUTCOME_COLS = select_oral_outcomes(oral_features)
print("Selected oral outcomes", len(ORAL_OUTCOME_COLS))
ORAL_OUTCOME_COLS[:80]


## Optional Oral Microbiome Structure Inspection
Run the next cell in the TRE example notebook or here if bulk loading still only returns raw read count. It prints the exact MetaPhlAn/HUMAnN path columns and the first bulk table shape/columns so the normalization rules can be tightened.


In [ ]:
# Optional diagnostics: run this if oral_features still only contains raw_read_count.
# It is intentionally read-only.
try:
    pl_dbg = make_phenoloader("oral_microbiome")
    oral_dbg = flatten_index(pl_dbg.dfs["oral_microbiome"])
    ds_dbg = phenoloader_dataset_dir(pl_dbg)
    print("oral_dbg shape:", oral_dbg.shape)
    print("path-like columns:")
    print([c for c in oral_dbg.columns if re.search(r"metaphlan|humann|parquet|arrow", str(c), re.I)])
    display(pl_dbg.dict.loc[pl_dbg.dict.index.astype(str).str.contains("metaphlan|humann|shannon|divers|species|genus", case=False, na=False)].head(120))
    for col in [c for c in oral_dbg.columns if re.search(r"metaphlan.*(genus|species).*parquet", str(c), re.I)]:
        raw = oral_dbg[col].dropna().astype(str).iloc[0]
        print("example", col, raw)
        for candidate in candidate_bulk_paths(raw, ds_dbg):
            if candidate.exists():
                tmp = read_bulk_file(candidate)
                print("readable:", candidate, "shape=", tmp.shape)
                display(flatten_index(tmp).head())
                print("columns:", flatten_index(tmp).columns.tolist()[:40])
                break
except Exception as exc:
    print("Diagnostics failed:", exc)


# 3. ADA Filtering Placeholder

No filtering is applied now. Later add eligibility/QC filters here: antibiotic use, oral sample QC, medication exclusions, temporal alignment, pregnancy, dental procedures, minimum diet logging days, etc.

In [ ]:
def apply_ada_filters(df: pd.DataFrame, *, label: str = "analysis") -> pd.DataFrame:
    print(f"ADA filtering placeholder for {label}: keeping {len(df):,} rows.")
    return df.copy()

# 4. Confounders

Edit `ACTIVE_CONFOUNDERS` in the configuration cell and rerun downward. These are loaded via `PhenoLoader('lifestyle_and_environment')` and `PhenoLoader('sociodemographics')` where available.

In [ ]:
CONFOUNDER_SPECS = {
    "age": [("lifestyle_and_environment", "age_sex", ["age"]), ("sociodemographics", "age_sex", ["age"]), ("oral_microbiome", "age_sex", ["age"])],
    "sex": [("lifestyle_and_environment", "age_sex", ["sex"]), ("sociodemographics", "age_sex", ["sex"]), ("oral_microbiome", "age_sex", ["sex"])],
    "bmi": [("lifestyle_and_environment", "lifestyle_and_environment", ["bmi", "body_mass_index"])],
    "smoking": [("lifestyle_and_environment", "lifestyle_and_environment", ["smoking_present_vs_past", "smoking_status", "smoking_current", "smoke"])],
    "alcohol": [("lifestyle_and_environment", "lifestyle_and_environment", ["alcohol_current_frequency", "alcohol_past_status", "alcohol", "drinking_frequency"])],
    "physical_activity": [("lifestyle_and_environment", "lifestyle_and_environment", ["activity_moderate_days_weekly", "activity_vigorous_days_weekly", "physical_activity", "exercise"])],
    "sleep_hours": [("lifestyle_and_environment", "lifestyle_and_environment", ["sleep_hours_daily", "sleep_hours", "sleep_duration"])],
    "education": [("sociodemographics", "initial_medical", ["education", "education_level"]), ("sociodemographics", "ukbb", ["education", "education_level"])],
}


def load_pheno_table(dataset: str, table: str) -> pd.DataFrame | None:
    try:
        from pheno_utils import PhenoLoader
        pl = PhenoLoader(dataset, errors="warn")
        if table in getattr(pl, "dfs", {}):
            return normalize_ids(flatten_index(pl.dfs[table]))
        return normalize_ids(flatten_index(pl[table]))
    except Exception as exc:
        print(f"Could not load {dataset}/{table}: {exc}")
        return None


def resolve_possible_columns(df: pd.DataFrame, possible_cols: list[str]) -> list[str]:
    exact = [c for c in possible_cols if c in df.columns]
    if exact:
        return exact
    lower_to_col = {str(c).lower(): c for c in df.columns}
    found = []
    for wanted in possible_cols:
        wanted_l = wanted.lower()
        for col_l, col in lower_to_col.items():
            if wanted_l == col_l or wanted_l in col_l:
                found.append(col)
                break
    return list(dict.fromkeys(found))


def collapse_participant_table(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    keep = [ID_COL] + [c for c in cols if c in df.columns]
    tmp = df[keep].copy()
    numeric = tmp.select_dtypes(include="number").columns.difference([ID_COL]).tolist()
    other = [c for c in tmp.columns if c not in [ID_COL] + numeric]
    parts = []
    if numeric:
        parts.append(tmp.groupby(ID_COL, as_index=False)[numeric].mean())
    for c in other:
        parts.append(tmp.groupby(ID_COL)[c].agg(lambda s: s.dropna().iloc[-1] if len(s.dropna()) else np.nan).reset_index())
    if not parts:
        return pd.DataFrame(columns=[ID_COL])
    out = parts[0]
    for part in parts[1:]:
        out = out.merge(part, on=ID_COL, how="outer")
    return out


def build_confounder_table(active: list[str]) -> pd.DataFrame:
    if CACHE_PATHS["confounders"].exists():
        cached = pd.read_pickle(CACHE_PATHS["confounders"])
        # Use cache only when columns satisfy requested active confounders reasonably.
        if all((c in cached.columns) or any(col.startswith(c + "__") for col in cached.columns) for c in active):
            print("Loaded cached confounders", cached.shape)
            return cached
    tables = []
    summary = []
    for alias in active:
        found = False
        for dataset, table, possible in CONFOUNDER_SPECS.get(alias, []):
            df = load_pheno_table(dataset, table)
            if df is None:
                continue
            present = resolve_possible_columns(df, possible)
            if not present:
                continue
            part = collapse_participant_table(df, present)
            rename = {present[0]: alias} if len(present) == 1 else {c: f"{alias}__{c}" for c in present}
            part = part.rename(columns=rename)
            tables.append(part)
            summary.append({"confounder": alias, "dataset": dataset, "table": table, "columns": present, "rows": len(part)})
            found = True
            break
        if not found:
            summary.append({"confounder": alias, "dataset": None, "table": None, "columns": [], "rows": 0})
    display(pd.DataFrame(summary))
    out = pd.DataFrame(columns=[ID_COL])
    for part in tables:
        out = part if out.empty else out.merge(part, on=ID_COL, how="outer")
    out = normalize_ids(out) if not out.empty else out
    out.to_pickle(CACHE_PATHS["confounders"])
    return out

confounders = build_confounder_table(ACTIVE_CONFOUNDERS)
confounders.head()

# 5. Build Analysis Tables

In [ ]:
def build_analysis_table(exposure_name: str) -> pd.DataFrame:
    exposure = exposure_tables[exposure_name].copy()
    merged = exposure.merge(oral_features[[ID_COL] + ORAL_OUTCOME_COLS], on=ID_COL, how="inner")
    if not confounders.empty:
        merged = merged.merge(confounders, on=ID_COL, how="left")
    return apply_ada_filters(merged, label=exposure_name)

analysis_tables = {name: build_analysis_table(name) for name in exposure_tables}
for name, df in analysis_tables.items():
    print(name, df.shape, df["exposure_group"].value_counts(dropna=False).to_dict(), df["decile_group"].value_counts(dropna=False).to_dict())

# 6. Statistical Tests

In [ ]:
def design_matrix_for_confounders(df: pd.DataFrame, confounder_cols: list[str]) -> tuple[np.ndarray | None, pd.DataFrame]:
    present = [c for c in confounder_cols if c in df.columns]
    if not present:
        return None, pd.DataFrame(index=df.index)
    x = df[present].copy()
    for c in x.columns:
        if pd.api.types.is_numeric_dtype(x[c]):
            x[c] = pd.to_numeric(x[c], errors="coerce").fillna(pd.to_numeric(x[c], errors="coerce").median())
        else:
            x[c] = x[c].astype("category").cat.add_categories(["missing"]).fillna("missing")
    x = pd.get_dummies(x, drop_first=True, dtype=float)
    x = x.loc[:, x.nunique(dropna=False) > 1]
    if x.empty:
        return None, x
    x.insert(0, "intercept", 1.0)
    return x.to_numpy(dtype=float), x


def residualize(y: pd.Series, df: pd.DataFrame, confounder_cols: list[str]) -> np.ndarray:
    yy = pd.to_numeric(y, errors="coerce").to_numpy(dtype=float)
    X, _ = design_matrix_for_confounders(df, confounder_cols)
    if X is None:
        return yy
    ok = np.isfinite(yy) & np.isfinite(X).all(axis=1)
    resid = np.full_like(yy, np.nan, dtype=float)
    if ok.sum() <= X.shape[1] + 2:
        return yy
    beta = np.linalg.lstsq(X[ok], yy[ok], rcond=None)[0]
    resid[ok] = yy[ok] - X[ok].dot(beta)
    return resid


def safe_kruskal(groups):
    groups = [np.asarray(g, dtype=float) for g in groups if len(g) >= MIN_GROUP_N]
    if len(groups) < 2:
        return np.nan
    pooled = np.concatenate(groups)
    pooled = pooled[np.isfinite(pooled)]
    if len(pooled) == 0:
        return np.nan
    if np.nanmax(pooled) == np.nanmin(pooled):
        return 1.0
    try:
        return stats.kruskal(*groups).pvalue
    except ValueError as exc:
        if "All numbers are identical" in str(exc):
            return 1.0
        raise


def safe_mannwhitneyu(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < MIN_GROUP_N or len(y) < MIN_GROUP_N:
        return np.nan
    pooled = np.concatenate([x, y])
    pooled = pooled[np.isfinite(pooled)]
    if len(pooled) == 0:
        return np.nan
    if np.nanmax(pooled) == np.nanmin(pooled):
        return 1.0
    return stats.mannwhitneyu(x, y, alternative="two-sided").pvalue




def summarize_numeric_group(values, prefix):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {f"{prefix}_mean": np.nan, f"{prefix}_median": np.nan, f"{prefix}_sd": np.nan}
    return {
        f"{prefix}_mean": float(np.mean(values)),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_sd": float(np.std(values, ddof=1)) if len(values) > 1 else np.nan,
    }


def low_high_tests(df, exposure_name, outcome, value_col=None):
    y_col = value_col or outcome
    low = finite_numeric(df.loc[df["exposure_group"] == "low", y_col])
    mid = finite_numeric(df.loc[df["exposure_group"] == "mid", y_col])
    high = finite_numeric(df.loc[df["exposure_group"] == "high", y_col])
    row = {"analysis": "tertile_low_mid_high", "exposure": exposure_name, "outcome": outcome, "n_low": len(low), "n_mid": len(mid), "n_high": len(high)}
    row.update(summarize_numeric_group(low, "low"))
    row.update(summarize_numeric_group(mid, "mid"))
    row.update(summarize_numeric_group(high, "high"))
    groups = [g for g in [low, mid, high] if len(g) >= MIN_GROUP_N]
    row["kruskal_p"] = safe_kruskal(groups)
    if len(low) >= MIN_GROUP_N and len(high) >= MIN_GROUP_N:
        p = safe_mannwhitneyu(low, high)
        d = cohens_d(low, high)
        ci_l, ci_h = cohens_d_ci(d, len(low), len(high))
        row.update({"low_vs_high_p": p, "cohens_d_high_minus_low": d, "cohens_d_ci_low": ci_l, "cohens_d_ci_high": ci_h, "cliffs_delta_high_minus_low": cliffs_delta(low, high), "median_diff_high_minus_low": np.median(high) - np.median(low)})
    else:
        row.update({"low_vs_high_p": np.nan, "cohens_d_high_minus_low": np.nan, "cohens_d_ci_low": np.nan, "cohens_d_ci_high": np.nan, "cliffs_delta_high_minus_low": np.nan, "median_diff_high_minus_low": np.nan})
    return row


def decile_tests(df, exposure_name, outcome, value_col=None):
    y_col = value_col or outcome
    bottom = finite_numeric(df.loc[df["decile_group"] == "bottom_10", y_col])
    top = finite_numeric(df.loc[df["decile_group"] == "top_10", y_col])
    row = {"analysis": "bottom10_vs_top10", "exposure": exposure_name, "outcome": outcome, "n_bottom_10": len(bottom), "n_top_10": len(top)}
    row.update(summarize_numeric_group(bottom, "bottom_10"))
    row.update(summarize_numeric_group(top, "top_10"))
    if len(bottom) >= MIN_GROUP_N and len(top) >= MIN_GROUP_N:
        p = safe_mannwhitneyu(bottom, top)
        d = cohens_d(bottom, top)
        ci_l, ci_h = cohens_d_ci(d, len(bottom), len(top))
        row.update({"bottom_vs_top_p": p, "cohens_d_top_minus_bottom": d, "cohens_d_ci_low": ci_l, "cohens_d_ci_high": ci_h, "cliffs_delta_top_minus_bottom": cliffs_delta(bottom, top), "median_diff_top_minus_bottom": np.median(top) - np.median(bottom)})
    else:
        row.update({"bottom_vs_top_p": np.nan, "cohens_d_top_minus_bottom": np.nan, "cohens_d_ci_low": np.nan, "cohens_d_ci_high": np.nan, "cliffs_delta_top_minus_bottom": np.nan, "median_diff_top_minus_bottom": np.nan})
    return row


def ols_regression(df, exposure_name, outcome, confounder_cols):
    cols = ["log1p_exposure_value", outcome] + [c for c in confounder_cols if c in df.columns]
    sub = df[cols].copy().dropna(subset=["log1p_exposure_value", outcome])
    row = {"analysis": "continuous_ols", "exposure": exposure_name, "outcome": outcome, "n": len(sub)}
    if len(sub) < MIN_GROUP_N or sub["log1p_exposure_value"].nunique() < 3:
        row.update({"beta": np.nan, "standardized_beta": np.nan, "p_value": np.nan, "r2": np.nan})
        return row
    X_conf, _ = design_matrix_for_confounders(sub, confounder_cols)
    x = pd.to_numeric(sub["log1p_exposure_value"], errors="coerce").to_numpy(dtype=float)
    y = pd.to_numeric(sub[outcome], errors="coerce").to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(sub)), x]) if X_conf is None else np.column_stack([X_conf, x])
    idx = X.shape[1] - 1
    ok = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X, y = X[ok], y[ok]
    if len(y) <= X.shape[1] + 2:
        row.update({"beta": np.nan, "standardized_beta": np.nan, "p_value": np.nan, "r2": np.nan})
        return row
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    pred = X.dot(beta)
    resid = y - pred
    dof = len(y) - X.shape[1]
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t_stat = beta[idx] / se[idx] if se[idx] > 0 else np.nan
    p = 2 * stats.t.sf(abs(t_stat), dof) if np.isfinite(t_stat) else np.nan
    ss_tot = ((y - y.mean()) @ (y - y.mean()))
    r2 = 1 - ((resid @ resid) / ss_tot) if ss_tot > 0 else np.nan
    std_beta = beta[idx] * np.nanstd(x) / np.nanstd(y) if np.nanstd(y) > 0 else np.nan
    row.update({"beta": beta[idx], "standardized_beta": std_beta, "p_value": p, "r2": r2})
    return row

confounder_cols = [c for c in confounders.columns if c != ID_COL]
tertile_rows, decile_rows, reg_rows = [], [], []
tertile_adj_rows, decile_adj_rows = [], []
for exposure_name, df in analysis_tables.items():
    for outcome in ORAL_OUTCOME_COLS:
        tertile_rows.append(low_high_tests(df, exposure_name, outcome))
        decile_rows.append(decile_tests(df, exposure_name, outcome))
        reg_rows.append(ols_regression(df, exposure_name, outcome, confounder_cols))
        adj = df.copy()
        adj[f"{outcome}__resid"] = residualize(adj[outcome], adj, confounder_cols)
        r = low_high_tests(adj, exposure_name, outcome, value_col=f"{outcome}__resid")
        r["analysis"] = "tertile_low_mid_high_adjusted_residual"
        tertile_adj_rows.append(r)
        r = decile_tests(adj, exposure_name, outcome, value_col=f"{outcome}__resid")
        r["analysis"] = "bottom10_vs_top10_adjusted_residual"
        decile_adj_rows.append(r)

tertile_results = pd.DataFrame(tertile_rows + tertile_adj_rows)
decile_results = pd.DataFrame(decile_rows + decile_adj_rows)
regression_results = pd.DataFrame(reg_rows)
tertile_results["q_value"] = p_adjust_bh(tertile_results["low_vs_high_p"])
decile_results["q_value"] = p_adjust_bh(decile_results["bottom_vs_top_p"])
regression_results["q_value"] = p_adjust_bh(regression_results["p_value"])
direction_audit_cols = [
    "exposure", "outcome", "n_low", "n_high",
    "low_mean", "high_mean", "low_median", "high_median",
    "cohens_d_high_minus_low", "cliffs_delta_high_minus_low", "median_diff_high_minus_low",
    "low_vs_high_p", "q_value",
]
print("Direction convention: positive Cohen's d / Cliff's delta / median difference = higher values in high exposure than low exposure.")
tertile_results.sort_values("low_vs_high_p")[direction_audit_cols].head(30)


# 7. Matplotlib Publication-Style Plots

White background, black axes/grids, consistent exposure-group colors, and p-values printed on plots. Save lines are commented out.

In [ ]:

def format_p(p):
    if p is None or not np.isfinite(p):
        return "p=NA"
    return f"p={p:.1e}" if p < 1e-4 else f"p={p:.4f}"


def pretty_label(value: str) -> str:
    return str(value).replace("_", " ")


def setup_axis(ax, title, x_title, y_title):
    ax.set_title(title, fontsize=13, color="black", pad=12)
    ax.set_xlabel(x_title, fontsize=12, color="black")
    ax.set_ylabel(y_title, fontsize=12, color="black")
    ax.set_facecolor("white")
    ax.figure.patch.set_facecolor("white")
    ax.grid(True, color="black", alpha=0.25, linewidth=0.7)
    for spine in ax.spines.values():
        spine.set_color("black")
    ax.tick_params(axis="both", colors="black", labelsize=11)


def p_value_for_plot(df, outcome, group_col, a, b):
    xa = finite_numeric(df.loc[df[group_col] == a, outcome])
    xb = finite_numeric(df.loc[df[group_col] == b, outcome])
    if len(xa) < MIN_GROUP_N or len(xb) < MIN_GROUP_N:
        return np.nan
    return stats.mannwhitneyu(xa, xb, alternative="two-sided").pvalue


def mean_ci(vals: np.ndarray, alpha: float = 0.05) -> tuple[float, float]:
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return np.nan, np.nan
    mean = float(np.mean(vals))
    if len(vals) < 2:
        return mean, np.nan
    ci = float(stats.t.ppf(1 - alpha / 2, len(vals) - 1) * stats.sem(vals, nan_policy="omit"))
    return mean, ci


def plot_group_distribution(df, exposure_name, outcome, group_col="exposure_group", kind="violin"):
    if group_col == "exposure_group":
        order, a, b = GROUP_ORDER, "low", "high"
        suffix = "low/mid/high exposure groups"
    else:
        order, a, b = ["bottom_10", "top_10"], "bottom_10", "top_10"
        suffix = "bottom/top 10% exposure groups"
    plot_df = df[df[group_col].isin(order)].copy()
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce").replace([np.inf, -np.inf], np.nan)
    values = [plot_df.loc[plot_df[group_col] == g, outcome].dropna().to_numpy(dtype=float) for g in order]
    p = p_value_for_plot(plot_df, outcome, group_col, a, b)
    fig, ax = plt.subplots(figsize=(7.8, 5.5), dpi=140)
    pos = np.arange(1, len(order) + 1)
    safe_values = [v if len(v) else np.array([np.nan]) for v in values]
    if kind == "violin":
        non_empty_positions = [pp for pp, vals in zip(pos, values) if len(vals) >= 2]
        non_empty_values = [vals for vals in values if len(vals) >= 2]
        if non_empty_values:
            parts = ax.violinplot(non_empty_values, positions=non_empty_positions, showmedians=True, showextrema=False)
            for body, ppos in zip(parts["bodies"], non_empty_positions):
                group = order[int(ppos) - 1]
                body.set_facecolor(GROUP_COLORS[group])
                body.set_edgecolor("black")
                body.set_alpha(0.65)
            parts["cmedians"].set_color("black")
    else:
        box = ax.boxplot(safe_values, positions=pos, patch_artist=True, widths=0.55, showfliers=False)
        for patch, group in zip(box["boxes"], order):
            patch.set_facecolor(GROUP_COLORS[group])
            patch.set_edgecolor("black")
            patch.set_alpha(0.65)
        for element in ["whiskers", "caps", "medians"]:
            for item in box[element]:
                item.set_color("black")

    rng = np.random.default_rng(123)
    means, sds, cis, ns = [], [], [], []
    for ppos, vals in zip(pos, values):
        ns.append(len(vals))
        mean, ci = mean_ci(vals)
        means.append(mean)
        sds.append(np.std(vals, ddof=1) if len(vals) > 1 else np.nan)
        cis.append(ci)
        if len(vals):
            ax.scatter(np.full(len(vals), ppos) + rng.normal(0, 0.045, len(vals)), vals, s=13, color="black", alpha=0.35, linewidths=0)
    ax.errorbar(pos - 0.055, means, yerr=sds, fmt="D", color="black", ecolor="black", capsize=6, markersize=4.5, label="mean +/- SD")
    ax.errorbar(pos + 0.055, means, yerr=cis, fmt="o", color="#b2182b", ecolor="#b2182b", capsize=6, markersize=4.5, label="mean 95% CI")
    ax.set_xticks(pos)
    ax.set_xticklabels([f"{g}\nn={n}" for g, n in zip(order, ns)])
    setup_axis(ax, f"{pretty_label(outcome)}\n{pretty_label(exposure_name)}: {suffix}", "Exposure group", pretty_label(outcome))
    ax.text(0.5, 0.98, f"{a} vs {b}: {format_p(p)}", transform=ax.transAxes, ha="center", va="top", bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.25"))
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    return fig, ax


def regression_line_with_ci(x: np.ndarray, y: np.ndarray, xs: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, object]:
    fit = stats.linregress(x, y)
    yhat = fit.intercept + fit.slope * xs
    pred = fit.intercept + fit.slope * x
    n = len(x)
    sxx = np.sum((x - x.mean()) ** 2)
    if n <= 2 or sxx <= 0:
        return yhat, np.full_like(xs, np.nan), np.full_like(xs, np.nan), fit
    resid = y - pred
    residual_se = np.sqrt(np.sum(resid ** 2) / (n - 2))
    tcrit = stats.t.ppf(0.975, n - 2)
    se_mean = residual_se * np.sqrt((1 / n) + ((xs - x.mean()) ** 2 / sxx))
    ci = tcrit * se_mean
    return yhat, yhat - ci, yhat + ci, fit


def plot_continuous_regression(df, exposure_name, outcome):
    plot_df = df[["log1p_exposure_value", outcome]].copy()
    plot_df["log1p_exposure_value"] = pd.to_numeric(plot_df["log1p_exposure_value"], errors="coerce")
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce")
    plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna()
    x = plot_df["log1p_exposure_value"].to_numpy(dtype=float)
    y = plot_df[outcome].to_numpy(dtype=float)

    fig, ax = plt.subplots(figsize=(7.8, 5.5), dpi=140)
    ax.scatter(x, y, s=18, color="#4d4d4d", alpha=0.45, linewidths=0)
    p_raw = np.nan
    slope = np.nan
    if len(plot_df) >= MIN_GROUP_N and np.unique(x).size > 2 and np.unique(y).size > 1:
        xs = np.linspace(np.nanmin(x), np.nanmax(x), 160)
        yhat, lo, hi, fit = regression_line_with_ci(x, y, xs)
        p_raw = fit.pvalue
        slope = fit.slope
        ax.plot(xs, yhat, color="#b2182b", linewidth=2.5, label="linear fit")
        ax.fill_between(xs, lo, hi, color="#b2182b", alpha=0.18, linewidth=0, label="95% CI")
    else:
        ax.text(0.5, 0.52, "Regression line not drawn\n(insufficient n or variance)", transform=ax.transAxes, ha="center", va="center", bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.3"))

    sub = regression_results[(regression_results["exposure"] == exposure_name) & (regression_results["outcome"] == outcome)] if "regression_results" in globals() else pd.DataFrame()
    txt = f"raw {format_p(p_raw)}\nraw slope={slope:.3g}\nn={len(plot_df)}"
    if not sub.empty:
        row = sub.iloc[0]
        txt += f"\nadjusted {format_p(float(row['p_value']))}\nadj beta={float(row['beta']):.3g}"
        if "q_value" in row.index:
            txt += f"\nFDR {format_p(float(row['q_value'])).replace('p=', 'q=')}"
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, ha="left", va="top", bbox=dict(facecolor="white", edgecolor="black", boxstyle="square,pad=0.25"))
    setup_axis(ax, f"Continuous exposure regression\n{pretty_label(exposure_name)} vs {pretty_label(outcome)}", "log1p nitrate-related food exposure (g)", pretty_label(outcome))
    if ax.get_legend_handles_labels()[0]:
        ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    return fig, ax


def plot_effect_size_forest(results, analysis, effect_col, title, top_n=25):
    sub = results[results["analysis"] == analysis].copy()
    sub[effect_col] = pd.to_numeric(sub[effect_col], errors="coerce")
    sub = sub[np.isfinite(sub[effect_col])]
    pcol = "low_vs_high_p" if "low_vs_high_p" in sub.columns else "bottom_vs_top_p"
    sub = sub.sort_values(pcol).head(top_n).iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9, max(5.5, 0.32 * len(sub) + 1.8)), dpi=140)
    y = np.arange(len(sub))
    labels = sub["exposure"].map(pretty_label) + " | " + sub["outcome"].map(pretty_label)
    xerr = np.vstack([sub[effect_col] - sub["cohens_d_ci_low"], sub["cohens_d_ci_high"] - sub[effect_col]])
    ax.errorbar(sub[effect_col], y, xerr=xerr, fmt="o", color="#2166ac", ecolor="black", capsize=4)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=9)
    setup_axis(ax, title, "Cohen's d with 95% CI", "Outcome")
    for yi, (_, row) in enumerate(sub.iterrows()):
        ax.text(row[effect_col], yi + 0.18, format_p(row[pcol]), fontsize=8, ha="center")
    fig.tight_layout()
    return fig, ax


In [ ]:
SELECTED_EXPOSURE = "vegetable_nitrate"
SELECTED_OUTCOME = ORAL_OUTCOME_COLS[0] if ORAL_OUTCOME_COLS else None
print("Selected", SELECTED_EXPOSURE, SELECTED_OUTCOME)
if SELECTED_OUTCOME:
    fig, ax = plot_group_distribution(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME, "exposure_group", "violin")
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"violin_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_group_distribution(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME, "exposure_group", "box")
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"box_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_group_distribution(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME, "decile_group", "violin")
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"violin_decile_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_continuous_regression(analysis_tables[SELECTED_EXPOSURE], SELECTED_EXPOSURE, SELECTED_OUTCOME)
    plt.show()
    # fig.savefig(OUTPUT_DIR / f"continuous_{SELECTED_EXPOSURE}_{SELECTED_OUTCOME}.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plot_effect_size_forest(tertile_results, "tertile_low_mid_high", "cohens_d_high_minus_low", "Effect sizes: high vs low exposure", top_n=25)
plt.show()
# fig.savefig(OUTPUT_DIR / "effect_size_forest_tertile_high_vs_low.png", dpi=300, bbox_inches="tight")

fig, ax = plot_effect_size_forest(decile_results, "bottom10_vs_top10", "cohens_d_top_minus_bottom", "Effect sizes: top 10% vs bottom 10% exposure", top_n=25)
plt.show()
# fig.savefig(OUTPUT_DIR / "effect_size_forest_decile_top_vs_bottom.png", dpi=300, bbox_inches="tight")

# 8. Rothia/Neisseria Focus And Top Increasing/Decreasing Features

This section expands the visualization beyond the preselected outcome list. It explicitly plots Rothia and Neisseria features and ranks the top oral microbiome features that increase or decrease in high nitrate/NO-axis exposure groups.


In [ ]:
# Rothia/Neisseria focus and top direction discovery.
# Uses all numeric oral_features columns, so it can discover taxa/pathways not included in ORAL_OUTCOME_COLS.

KEY_NITRATE_TAXA = ["rothia", "neisseria"]
DISCOVERY_EXPOSURES = [name for name in ["vegetable_nitrate", "processed_nitrite", "nitroso_axis", "arginine_no_axis"] if name in exposure_tables]
DISCOVERY_EXPOSURE = "vegetable_nitrate" if "vegetable_nitrate" in exposure_tables else DISCOVERY_EXPOSURES[0]
DISCOVERY_TOP_N = 10


def oral_discovery_feature_columns(oral: pd.DataFrame) -> list[str]:
    numeric_cols = [c for c in oral.select_dtypes(include="number").columns if c != ID_COL]
    exclude_tokens = ["matched_feature_count", "raw_read_count"]
    return [c for c in numeric_cols if not any(tok in c for tok in exclude_tokens)]


def build_discovery_analysis_table(exposure_name: str) -> pd.DataFrame:
    feature_cols = oral_discovery_feature_columns(oral_features)
    exposure = exposure_tables[exposure_name].copy()
    merged = exposure.merge(oral_features[[ID_COL] + feature_cols], on=ID_COL, how="inner")
    if not confounders.empty:
        # Keep confounders for optional downstream interaction/regression, but ranking below is unadjusted high vs low.
        add_cols = [c for c in confounders.columns if c != ID_COL and c not in merged.columns]
        if add_cols:
            merged = merged.merge(confounders[[ID_COL] + add_cols], on=ID_COL, how="left")
    return apply_ada_filters(merged, label=f"discovery_{exposure_name}")


def rank_increasing_decreasing_features(exposure_name: str, pattern: str | None = None) -> pd.DataFrame:
    df = build_discovery_analysis_table(exposure_name)
    feature_cols = oral_discovery_feature_columns(oral_features)
    if pattern:
        rx = re.compile(pattern, re.I)
        feature_cols = [c for c in feature_cols if rx.search(c)]
    rows = []
    for outcome in feature_cols:
        if outcome not in df.columns:
            continue
        row = low_high_tests(df, exposure_name, outcome)
        rows.append(row)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["q_value"] = p_adjust_bh(out["low_vs_high_p"])
    out["direction"] = np.where(out["cohens_d_high_minus_low"] > 0, "increases_with_high_exposure", "decreases_with_high_exposure")
    return out.sort_values("cohens_d_high_minus_low", ascending=False)


def find_taxon_feature_columns(taxon: str, oral: pd.DataFrame, prefer_summary: bool = True) -> list[str]:
    numeric_cols = oral_discovery_feature_columns(oral)
    rx = re.compile(taxon, re.I)
    cols = [c for c in numeric_cols if rx.search(c)]
    if prefer_summary:
        def score(c):
            score_val = 0
            if "nitrate_reducer" in c:
                score_val += 100
            if "_present" in c:
                score_val -= 20
            if "oral_metaphlan_genus" in c:
                score_val += 30
            if "oral_metaphlan_species" in c:
                score_val += 20
            if "oral_metaphlan_family" in c:
                score_val += 10
            if c.endswith(f"_{taxon}") or f"_{taxon}_" in c:
                score_val += 5
            return score_val
        cols = sorted(cols, key=score, reverse=True)
    return cols


def plot_key_taxa_for_exposure(exposure_name: str, taxa: list[str] = KEY_NITRATE_TAXA, max_cols_per_taxon: int = 2):
    df = build_discovery_analysis_table(exposure_name)
    plotted = []
    for taxon in taxa:
        cols = find_taxon_feature_columns(taxon, oral_features)[:max_cols_per_taxon]
        print(f"{taxon}: plotting {cols}")
        for outcome in cols:
            if outcome not in df.columns:
                continue
            fig, ax = plot_group_distribution(df, exposure_name, outcome, "exposure_group", "box")
            plt.show()
            # fig.savefig(OUTPUT_DIR / f"key_taxon_box_{exposure_name}_{outcome}.png", dpi=300, bbox_inches="tight")
            fig, ax = plot_continuous_regression(df, exposure_name, outcome)
            plt.show()
            # fig.savefig(OUTPUT_DIR / f"key_taxon_continuous_{exposure_name}_{outcome}.png", dpi=300, bbox_inches="tight")
            plotted.append(outcome)
    return plotted


def plot_direction_bar(rank_df: pd.DataFrame, title: str, top_n: int = 10):
    sub = rank_df.copy()
    sub = sub[np.isfinite(pd.to_numeric(sub["cohens_d_high_minus_low"], errors="coerce"))]
    if sub.empty:
        print("No features to plot")
        return None, None
    sub = sub.head(top_n).iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(9.5, max(4.8, 0.42 * len(sub) + 1.5)), dpi=140)
    y = np.arange(len(sub))
    colors = ["#b2182b" if v > 0 else "#2166ac" for v in sub["cohens_d_high_minus_low"]]
    labels = [pretty_label(x)[-85:] for x in sub["outcome"]]
    ax.barh(y, sub["cohens_d_high_minus_low"], color=colors, edgecolor="black", alpha=0.8)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    setup_axis(ax, title, "Cohen's d: high exposure minus low exposure", "Oral microbiome feature")
    for yi, (_, row) in enumerate(sub.iterrows()):
        ax.text(row["cohens_d_high_minus_low"], yi, f" {format_p(row['low_vs_high_p'])}", va="center", fontsize=8)
    fig.tight_layout()
    return fig, ax


# Explicit Rothia and Neisseria visualization for the selected nitrate-relevant exposure.
print("Discovery exposure:", DISCOVERY_EXPOSURE)
key_taxa_plotted = plot_key_taxa_for_exposure(DISCOVERY_EXPOSURE, KEY_NITRATE_TAXA, max_cols_per_taxon=2)
print("Key taxa plotted:", key_taxa_plotted)

# Rank all oral features by high-vs-low effect direction for each nitrate/NO-axis exposure.
discovery_rankings = {}
for exposure_name in DISCOVERY_EXPOSURES:
    rank_df = rank_increasing_decreasing_features(exposure_name)
    discovery_rankings[exposure_name] = rank_df
    print("\n", "=" * 90)
    print(exposure_name, "ranked features:", rank_df.shape)
    increasing = rank_df.sort_values("cohens_d_high_minus_low", ascending=False).head(DISCOVERY_TOP_N)
    decreasing = rank_df.sort_values("cohens_d_high_minus_low", ascending=True).head(DISCOVERY_TOP_N)
    print("Top increasing with high exposure")
    display(increasing[["exposure", "outcome", "n_low", "n_high", "low_mean", "high_mean", "cohens_d_high_minus_low", "cliffs_delta_high_minus_low", "median_diff_high_minus_low", "low_vs_high_p", "q_value"]])
    print("Top decreasing with high exposure")
    display(decreasing[["exposure", "outcome", "n_low", "n_high", "low_mean", "high_mean", "cohens_d_high_minus_low", "cliffs_delta_high_minus_low", "median_diff_high_minus_low", "low_vs_high_p", "q_value"]])

    fig, ax = plot_direction_bar(increasing, f"Top {DISCOVERY_TOP_N} increasing oral features: {pretty_label(exposure_name)}", top_n=DISCOVERY_TOP_N)
    if fig is not None:
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"top_increasing_{exposure_name}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_direction_bar(decreasing.sort_values("cohens_d_high_minus_low", ascending=False), f"Top {DISCOVERY_TOP_N} decreasing oral features: {pretty_label(exposure_name)}", top_n=DISCOVERY_TOP_N)
    if fig is not None:
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"top_decreasing_{exposure_name}.png", dpi=300, bbox_inches="tight")


# 9. Effect Modification By Sex, Age, And Relationship Status

This section asks whether the diet-to-oral-microbiome association differs by sex, becomes stronger with age, or is stronger among people classified as single/not partnered where relationship-status fields are available. It does not replace the main tests above; it adds stratified and interaction analyses.


In [ ]:
# Effect modification / subgroup analysis.
# This section is intentionally independent of ACTIVE_CONFOUNDERS so sex/age/relationship status
# are available for subgroup analyses even if you remove them from the confounder list.

EFFECT_MODIFIER_SPECS = {
    "age": [
        ("oral_microbiome", "age_sex", ["age"]),
        ("lifestyle_and_environment", "age_sex", ["age"]),
        ("sociodemographics", "age_sex", ["age"]),
    ],
    "sex": [
        ("oral_microbiome", "age_sex", ["sex"]),
        ("lifestyle_and_environment", "age_sex", ["sex"]),
        ("sociodemographics", "age_sex", ["sex"]),
    ],
    "relationship_status": [
        ("sociodemographics", "initial_medical", [
            "marital_status", "relationship_status", "living_with_partner", "living_with_spouse",
            "single", "is_single", "household_status", "household_composition", "living_arrangements",
            "family_status", "partner_status", "current_relationship_status",
        ]),
        ("sociodemographics", "ukbb", [
            "marital_status", "relationship_status", "living_with_partner", "living_with_spouse",
            "single", "is_single", "household_status", "household_composition", "living_arrangements",
            "family_status", "partner_status", "current_relationship_status",
        ]),
        ("lifestyle_and_environment", "lifestyle_and_environment", [
            "marital_status", "relationship_status", "living_with_partner", "living_with_spouse",
            "single", "is_single", "household_status", "household_composition", "living_arrangements",
            "family_status", "partner_status", "current_relationship_status",
        ]),
    ],
}


def build_effect_modifier_table() -> pd.DataFrame:
    parts = []
    summary = []
    for alias, specs in EFFECT_MODIFIER_SPECS.items():
        found = False
        for dataset, table, possible in specs:
            df = load_pheno_table(dataset, table)
            if df is None:
                continue
            present = resolve_possible_columns(df, possible)
            if not present:
                continue
            part = collapse_participant_table(df, present)
            # Prefer the first resolved column for a clean modifier alias; keep source column for relationship yes/no direction.
            source_col = present[0]
            part = part.rename(columns={source_col: alias})[[ID_COL, alias]]
            if alias == "relationship_status":
                part["relationship_status_source_col"] = source_col
            parts.append(part)
            summary.append({"modifier": alias, "dataset": dataset, "table": table, "column_used": source_col, "other_matches": present[1:], "rows": len(part)})
            found = True
            break
        if not found:
            summary.append({"modifier": alias, "dataset": None, "table": None, "column_used": None, "other_matches": [], "rows": 0})
    display(pd.DataFrame(summary))

    out = pd.DataFrame(columns=[ID_COL])
    for part in parts:
        out = part if out.empty else out.merge(part, on=ID_COL, how="outer")
    return normalize_ids(out) if not out.empty else out


def standardize_sex_value(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if s in {"f", "female", "woman", "women", "0.0", "0"}:
        return "female"
    if s in {"m", "male", "man", "men", "1.0", "1"}:
        return "male"
    if "female" in s or "woman" in s:
        return "female"
    if "male" in s or "man" in s:
        return "male"
    return str(value)


def standardize_relationship_value(value, source_col=None):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    src = str(source_col or "").strip().lower()
    single_tokens = ["single", "never married", "divorced", "separated", "widowed", "no partner", "not living with", "alone"]
    partnered_tokens = ["married", "partner", "spouse", "cohab", "living with", "civil partnership", "couple"]

    truthy = s in {"1", "1.0", "true", "yes", "y"}
    falsey = s in {"0", "0.0", "false", "no", "n"}
    if truthy or falsey:
        if "single" in src or "alone" in src or "not_partner" in src or "not partner" in src:
            return "single_or_not_partnered" if truthy else "partnered_or_not_single"
        if "partner" in src or "spouse" in src or "married" in src or "cohab" in src:
            return "partnered_or_not_single" if truthy else "single_or_not_partnered"
        return "binary_true" if truthy else "binary_false"

    if any(tok in s for tok in single_tokens):
        return "single_or_not_partnered"
    if any(tok in s for tok in partnered_tokens):
        return "partnered_or_not_single"
    return str(value)


def add_effect_modifiers_to_analysis_tables(analysis_tables: dict[str, pd.DataFrame], modifiers: pd.DataFrame) -> dict[str, pd.DataFrame]:
    out = {}
    for exposure_name, df in analysis_tables.items():
        merged = df.copy()
        add_cols = [c for c in modifiers.columns if c != ID_COL and c not in merged.columns]
        if add_cols:
            merged = merged.merge(modifiers[[ID_COL] + add_cols], on=ID_COL, how="left")
        if "sex" in merged.columns:
            merged["sex_group"] = merged["sex"].map(standardize_sex_value)
        if "age" in merged.columns:
            merged["age"] = pd.to_numeric(merged["age"], errors="coerce")
            try:
                merged["age_band"] = pd.qcut(merged["age"], q=3, labels=["younger", "middle_age", "older"], duplicates="drop")
            except Exception:
                merged["age_band"] = pd.cut(merged["age"], bins=3, labels=["younger", "middle_age", "older"])
        if "relationship_status" in merged.columns:
            source_col = merged["relationship_status_source_col"].dropna().iloc[0] if "relationship_status_source_col" in merged.columns and merged["relationship_status_source_col"].notna().any() else None
            merged["relationship_group"] = merged["relationship_status"].map(lambda v: standardize_relationship_value(v, source_col=source_col))
        out[exposure_name] = merged
    return out


def subgroup_low_high_tests(df: pd.DataFrame, exposure_name: str, outcome: str, subgroup_col: str) -> list[dict]:
    rows = []
    if subgroup_col not in df.columns:
        return rows
    for subgroup, sub in df.dropna(subset=[subgroup_col]).groupby(subgroup_col):
        if len(sub) < MIN_GROUP_N * 2:
            continue
        row = low_high_tests(sub, exposure_name, outcome)
        row["analysis"] = f"effect_modifier_{subgroup_col}_low_high"
        row["modifier"] = subgroup_col
        row["modifier_level"] = str(subgroup)
        row["n_subgroup"] = len(sub)
        rows.append(row)
    return rows


def interaction_ols(df: pd.DataFrame, exposure_name: str, outcome: str, modifier_col: str, confounder_cols: list[str]) -> dict:
    row = {"analysis": f"interaction_{modifier_col}", "exposure": exposure_name, "outcome": outcome, "modifier": modifier_col}
    if modifier_col not in df.columns:
        row.update({"n": 0, "interaction_beta": np.nan, "interaction_p": np.nan, "main_exposure_beta": np.nan})
        return row

    cols = ["log1p_exposure_value", outcome, modifier_col] + [c for c in confounder_cols if c in df.columns and c != modifier_col]
    sub = df[cols].copy().dropna(subset=["log1p_exposure_value", outcome, modifier_col])
    row["n"] = len(sub)
    if len(sub) < MIN_GROUP_N or sub["log1p_exposure_value"].nunique() < 3:
        row.update({"interaction_beta": np.nan, "interaction_p": np.nan, "main_exposure_beta": np.nan})
        return row

    y = pd.to_numeric(sub[outcome], errors="coerce")
    x = pd.to_numeric(sub["log1p_exposure_value"], errors="coerce")
    m_raw = sub[modifier_col]

    if pd.api.types.is_numeric_dtype(m_raw):
        m = pd.to_numeric(m_raw, errors="coerce")
        if m.notna().sum() == 0 or m.nunique(dropna=True) < 2:
            row.update({"interaction_beta": np.nan, "interaction_p": np.nan, "main_exposure_beta": np.nan})
            return row
        m = (m - m.mean()) / m.std(ddof=0) if m.std(ddof=0) > 0 else m - m.mean()
        X = pd.DataFrame({"intercept": 1.0, "x": x, "modifier": m, "x_modifier": x * m})
    else:
        dummies = pd.get_dummies(m_raw.astype("category"), prefix=modifier_col, drop_first=True, dtype=float)
        if dummies.empty:
            row.update({"interaction_beta": np.nan, "interaction_p": np.nan, "main_exposure_beta": np.nan})
            return row
        X = pd.DataFrame({"intercept": 1.0, "x": x})
        X = pd.concat([X, dummies], axis=1)
        for c in dummies.columns:
            X[f"x__{c}"] = x * dummies[c]

    extra_confounders = [c for c in confounder_cols if c in sub.columns and c != modifier_col]
    if extra_confounders:
        _, conf_design = design_matrix_for_confounders(sub, extra_confounders)
        if not conf_design.empty:
            conf_design = conf_design.drop(columns=["intercept"], errors="ignore")
            X = pd.concat([X, conf_design.reset_index(drop=True)], axis=1)

    X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(y, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ok = y.notna() & X.notna().all(axis=1)
    Xn = X.loc[ok].to_numpy(dtype=float)
    yn = y.loc[ok].to_numpy(dtype=float)
    if len(yn) <= Xn.shape[1] + 2:
        row.update({"interaction_beta": np.nan, "interaction_p": np.nan, "main_exposure_beta": np.nan})
        return row

    beta = np.linalg.lstsq(Xn, yn, rcond=None)[0]
    pred = Xn.dot(beta)
    resid = yn - pred
    dof = len(yn) - Xn.shape[1]
    sigma2 = (resid @ resid) / dof
    cov = sigma2 * np.linalg.pinv(Xn.T @ Xn)
    se = np.sqrt(np.diag(cov))
    names = X.columns.tolist()
    interaction_cols = [i for i, name in enumerate(names) if name.startswith("x_modifier") or name.startswith("x__")]
    if not interaction_cols:
        row.update({"interaction_beta": np.nan, "interaction_p": np.nan, "main_exposure_beta": beta[names.index("x")] if "x" in names else np.nan})
        return row

    if len(interaction_cols) == 1:
        idx = interaction_cols[0]
        t_stat = beta[idx] / se[idx] if se[idx] > 0 else np.nan
        p = 2 * stats.t.sf(abs(t_stat), dof) if np.isfinite(t_stat) else np.nan
        row.update({"interaction_beta": beta[idx], "interaction_p": p, "main_exposure_beta": beta[names.index("x")] if "x" in names else np.nan})
    else:
        # Omnibus F-test for multi-level categorical interaction terms.
        keep = [i for i in range(Xn.shape[1]) if i not in interaction_cols]
        X_reduced = Xn[:, keep]
        beta_r = np.linalg.lstsq(X_reduced, yn, rcond=None)[0]
        resid_r = yn - X_reduced.dot(beta_r)
        rss_full = float(resid @ resid)
        rss_reduced = float(resid_r @ resid_r)
        df_num = len(interaction_cols)
        df_den = dof
        f_stat = ((rss_reduced - rss_full) / df_num) / (rss_full / df_den) if rss_full > 0 else np.nan
        p = stats.f.sf(f_stat, df_num, df_den) if np.isfinite(f_stat) else np.nan
        row.update({"interaction_beta": np.nanmean(beta[interaction_cols]), "interaction_p": p, "main_exposure_beta": beta[names.index("x")] if "x" in names else np.nan})
    return row


effect_modifiers = build_effect_modifier_table()
analysis_tables_em = add_effect_modifiers_to_analysis_tables(analysis_tables, effect_modifiers)

modifier_cols = [c for c in ["sex_group", "age_band", "relationship_group"] if any(c in df.columns for df in analysis_tables_em.values())]
interaction_cols = [c for c in ["sex_group", "age", "relationship_group"] if any(c in df.columns for df in analysis_tables_em.values())]
print("Modifier columns for stratified tests:", modifier_cols)
print("Modifier columns for interaction tests:", interaction_cols)

subgroup_rows = []
interaction_rows = []
for exposure_name, df in analysis_tables_em.items():
    for outcome in ORAL_OUTCOME_COLS:
        for mod in modifier_cols:
            subgroup_rows.extend(subgroup_low_high_tests(df, exposure_name, outcome, mod))
        for mod in interaction_cols:
            interaction_rows.append(interaction_ols(df, exposure_name, outcome, mod, confounder_cols))

subgroup_effect_results = pd.DataFrame(subgroup_rows)
interaction_effect_results = pd.DataFrame(interaction_rows)
if not subgroup_effect_results.empty and "low_vs_high_p" in subgroup_effect_results.columns:
    subgroup_effect_results["q_value"] = p_adjust_bh(subgroup_effect_results["low_vs_high_p"])
if not interaction_effect_results.empty and "interaction_p" in interaction_effect_results.columns:
    interaction_effect_results["interaction_q_value"] = p_adjust_bh(interaction_effect_results["interaction_p"])

print("Subgroup rows:", subgroup_effect_results.shape)
print("Interaction rows:", interaction_effect_results.shape)
display(subgroup_effect_results.sort_values("low_vs_high_p", na_position="last").head(30) if not subgroup_effect_results.empty else subgroup_effect_results)
display(interaction_effect_results.sort_values("interaction_p", na_position="last").head(30) if not interaction_effect_results.empty else interaction_effect_results)


## Effect-Modification Plots

Use `SELECTED_EXPOSURE` and `SELECTED_OUTCOME` from the plotting section, or manually set them here before running the plots.


In [ ]:
def plot_effect_modifier_group_means(df, exposure_name, outcome, modifier_col):
    if modifier_col not in df.columns:
        print(f"{modifier_col} is not available for {exposure_name}")
        return None, None
    plot_df = df[df["exposure_group"].isin(GROUP_ORDER)].copy()
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce")
    plot_df = plot_df.dropna(subset=[outcome, modifier_col])
    levels = [x for x in plot_df[modifier_col].dropna().astype(str).unique() if str(x).lower() != "nan"]
    levels = sorted(levels)[:8]
    if not levels:
        print(f"No levels found for {modifier_col}")
        return None, None

    fig, ax = plt.subplots(figsize=(8.6, 5.5), dpi=140)
    x = np.arange(len(GROUP_ORDER))
    offsets = np.linspace(-0.18, 0.18, len(levels)) if len(levels) > 1 else [0]
    for offset, level in zip(offsets, levels):
        means, cis, ns = [], [], []
        for group in GROUP_ORDER:
            vals = finite_numeric(plot_df.loc[(plot_df["exposure_group"] == group) & (plot_df[modifier_col].astype(str) == level), outcome])
            mean, ci = mean_ci(vals)
            means.append(mean)
            cis.append(ci)
            ns.append(len(vals))
        ax.errorbar(x + offset, means, yerr=cis, marker="o", linewidth=2, capsize=5, label=f"{level} (n={sum(ns)})")
    ax.set_xticks(x)
    ax.set_xticklabels(GROUP_ORDER)
    setup_axis(ax, f"{pretty_label(outcome)} by {pretty_label(exposure_name)} and {pretty_label(modifier_col)}", "Exposure group", pretty_label(outcome))
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    return fig, ax


def plot_age_interaction_scatter(df, exposure_name, outcome):
    if "age" not in df.columns:
        print("age is not available")
        return None, None
    plot_df = df[["log1p_exposure_value", "age", outcome]].copy()
    plot_df["log1p_exposure_value"] = pd.to_numeric(plot_df["log1p_exposure_value"], errors="coerce")
    plot_df["age"] = pd.to_numeric(plot_df["age"], errors="coerce")
    plot_df[outcome] = pd.to_numeric(plot_df[outcome], errors="coerce")
    plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna()
    if len(plot_df) < MIN_GROUP_N:
        print("Not enough data for age interaction plot")
        return None, None

    fig, ax = plt.subplots(figsize=(8.2, 5.5), dpi=140)
    sc = ax.scatter(plot_df["log1p_exposure_value"], plot_df[outcome], c=plot_df["age"], cmap="viridis", s=18, alpha=0.55, linewidths=0)
    cb = fig.colorbar(sc, ax=ax)
    cb.set_label("Age")
    for label, sub in plot_df.groupby(pd.qcut(plot_df["age"], q=3, labels=["younger", "middle", "older"], duplicates="drop")):
        if len(sub) >= MIN_GROUP_N and sub["log1p_exposure_value"].nunique() > 2 and sub[outcome].nunique() > 1:
            x = sub["log1p_exposure_value"].to_numpy(dtype=float)
            y = sub[outcome].to_numpy(dtype=float)
            xs = np.linspace(x.min(), x.max(), 100)
            yhat, lo, hi, fit = regression_line_with_ci(x, y, xs)
            ax.plot(xs, yhat, linewidth=2, label=f"{label}: slope={fit.slope:.3g}, {format_p(fit.pvalue)}")
    setup_axis(ax, f"Age-modified continuous association\n{pretty_label(exposure_name)} vs {pretty_label(outcome)}", "log1p nitrate-related food exposure (g)", pretty_label(outcome))
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    return fig, ax


SELECTED_EXPOSURE_EM = SELECTED_EXPOSURE if "SELECTED_EXPOSURE" in globals() else "vegetable_nitrate"
SELECTED_OUTCOME_EM = SELECTED_OUTCOME if "SELECTED_OUTCOME" in globals() else (ORAL_OUTCOME_COLS[0] if ORAL_OUTCOME_COLS else None)
print("Selected effect-modification plot:", SELECTED_EXPOSURE_EM, SELECTED_OUTCOME_EM)
if SELECTED_OUTCOME_EM and SELECTED_EXPOSURE_EM in analysis_tables_em:
    for modifier in ["sex_group", "age_band", "relationship_group"]:
        fig, ax = plot_effect_modifier_group_means(analysis_tables_em[SELECTED_EXPOSURE_EM], SELECTED_EXPOSURE_EM, SELECTED_OUTCOME_EM, modifier)
        if fig is not None:
            plt.show()
            # fig.savefig(OUTPUT_DIR / f"effect_modifier_{modifier}_{SELECTED_EXPOSURE_EM}_{SELECTED_OUTCOME_EM}.png", dpi=300, bbox_inches="tight")
    fig, ax = plot_age_interaction_scatter(analysis_tables_em[SELECTED_EXPOSURE_EM], SELECTED_EXPOSURE_EM, SELECTED_OUTCOME_EM)
    if fig is not None:
        plt.show()
        # fig.savefig(OUTPUT_DIR / f"age_interaction_{SELECTED_EXPOSURE_EM}_{SELECTED_OUTCOME_EM}.png", dpi=300, bbox_inches="tight")


# 10. Summary Tables


In [ ]:
tertile_summary = tertile_results.sort_values(["q_value", "low_vs_high_p"], na_position="last")
decile_summary = decile_results.sort_values(["q_value", "bottom_vs_top_p"], na_position="last")
regression_summary = regression_results.sort_values(["q_value", "p_value"], na_position="last")
tertile_summary.head(50)

In [ ]:
decile_summary.head(50)

In [ ]:
regression_summary.head(50)

In [ ]:
compact_tertile = tertile_summary.rename(columns={"low_vs_high_p": "primary_p", "cohens_d_high_minus_low": "effect_size", "median_diff_high_minus_low": "median_difference"}).copy()
compact_tertile["test_family"] = "low_mid_high"
compact_decile = decile_summary.rename(columns={"bottom_vs_top_p": "primary_p", "cohens_d_top_minus_bottom": "effect_size", "median_diff_top_minus_bottom": "median_difference"}).copy()
compact_decile["test_family"] = "bottom_top_10"
compact_reg = regression_summary.rename(columns={"p_value": "primary_p", "standardized_beta": "effect_size"}).copy()
compact_reg["test_family"] = "continuous_regression"
compact_reg["median_difference"] = np.nan
common = ["test_family", "analysis", "exposure", "outcome", "primary_p", "q_value", "effect_size", "median_difference"]
final_summary_table = pd.concat([
    compact_tertile[[c for c in common if c in compact_tertile.columns]],
    compact_decile[[c for c in common if c in compact_decile.columns]],
    compact_reg[[c for c in common if c in compact_reg.columns]],
], ignore_index=True, sort=False).sort_values(["q_value", "primary_p"], na_position="last")
final_summary_table.head(100)

In [ ]:
# Optional table export after finalizing settings.
# tertile_summary.to_csv(OUTPUT_DIR / "nitrate_oral_tertile_results.csv", index=False)
# decile_summary.to_csv(OUTPUT_DIR / "nitrate_oral_decile_results.csv", index=False)
# regression_summary.to_csv(OUTPUT_DIR / "nitrate_oral_continuous_regression_results.csv", index=False)
# final_summary_table.to_csv(OUTPUT_DIR / "nitrate_oral_final_summary_table.csv", index=False)